# EDA + Feature Engineering

**Input:** NB00's `panel_long.parquet` and `macro_q.parquet`, through `container.panel_store()`

**Output:** `data/features/features_h{1..4}.parquet` (all companies pooled, one row per origin quarter) and one W&B run per target variable

Every collaborator comes from the DI container; this notebook only orchestrates (spec R10).
Specs: `docs/specs/eda-feature-engineering.md`, `docs/specs/feature-diagnostics-services.md`.

## 1. Load panels

One frame per company with macro columns and NB00 flags joined on; projected rows dropped.
COP is excluded until its revenue ingestion defect is fixed (Q4 2010 is −81,500, which log growth cannot take).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from app.injections import configure_container
from app.services.features import FeatureGroup, TargetArm, TargetTransformer
from app.services.tracking import RunConfig, run_name

container = configure_container()
tracker = container.experiment_tracker()
diagnostics_service = container.diagnostics_service()
feature_service = container.feature_service()
feature_store = container.feature_store()

EXCLUDED = ("COP",)
NB00_COMPANIES = 60
QUARTERS = 81
panels = {
    ticker: panel
    for ticker, panel in container.panel_store().load_consolidated().items()
    if ticker not in EXCLUDED
}
n_rows = sum(len(panel) for panel in panels.values())

assert len(panels) == NB00_COMPANIES - len(EXCLUDED)  # A1
assert all(len(panel) == QUARTERS for panel in panels.values())  # A1
assert all({"covid", "structural_break"} <= set(p.columns) for p in panels.values())

TARGETS = [
    "revenue_usd_m",
    "operating_income_usd_m",
    "ebitda_usd_m",
    "free_cash_flow_usd_m",
]

## 2. Diagnostics: stationarity, seasonality, trend

One record per (company, variable): differencing order on levels and logs (ADF and KPSS must agree), STL seasonal and trend strength, Ljung-Box, Box-Cox λ, CV. Short, gapped or constant series yield a status, never an exception.

In [ ]:
diagnostics = diagnostics_service.diagnose_panel(panels, TARGETS)
revenue = diagnostics.query("variable == 'revenue_usd_m'").set_index("ticker")

SHORT_HISTORIES = {"TSLA": 72, "BBY": 78, "ADBE": 80, "ORCL": 80}
assert len(diagnostics) == len(panels) * len(TARGETS)  # A2
assert revenue.loc[list(SHORT_HISTORIES), "n_obs"].tolist() == list(
    SHORT_HISTORIES.values()
)  # A2
assert revenue["n_obs"].drop(list(SHORT_HISTORIES)).eq(QUARTERS).all()  # A2

regimes = diagnostics_service.seasonality_regimes(diagnostics, variable="revenue_usd_m")
sector_sensitivity = (
    diagnostics_service
    .growth_macro_correlations(panels, variable="revenue_usd_m")
    .groupby("sector")
    .mean(numeric_only=True)
)
diagnostics.groupby(["variable", "status"]).size()

In [ ]:
seasonal_figure, seasonal_axes = plt.subplots(figsize=(8, 4))
revenue["seasonal_strength"].plot.hist(bins=20, ax=seasonal_axes)
seasonal_axes.set(
    title="Revenue seasonal strength (STL, period 4)",
    xlabel="seasonal strength",
    ylabel="companies",
)

differencing_orders = pd.crosstab(
    revenue["d_levels"], revenue["d_log"], rownames=["d_levels"], colnames=["d_log"]
)

sensitivity_figure, sensitivity_axes = plt.subplots(figsize=(10, 6))
heatmap = sensitivity_axes.imshow(
    sector_sensitivity.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1
)
sensitivity_axes.set_xticks(
    range(len(sector_sensitivity.columns)),
    sector_sensitivity.columns,
    rotation=45,
    ha="right",
)
sensitivity_axes.set_yticks(
    range(len(sector_sensitivity.index)), sector_sensitivity.index
)
sensitivity_axes.set_title("Revenue YoY growth vs macro: mean correlation by sector")
sensitivity_figure.colorbar(heatmap, ax=sensitivity_axes)
sensitivity_figure.tight_layout()

In [ ]:
for variable in TARGETS:
    config = RunConfig(panel_size=len(panels), n_rows=n_rows, target_variable=variable)
    with tracker.start_run(
        run_name(notebook="nb01", model="eda", variable=variable),
        config,
        job_type="eda",
    ) as run:
        run.log_table("diagnostics", diagnostics.query("variable == @variable"))
        if variable == "revenue_usd_m":
            run.log_table("seasonality_regimes", regimes.reset_index())
            run.log_table("differencing_orders", differencing_orders.reset_index())
            run.log_table(
                "macro_sensitivity_by_sector", sector_sensitivity.reset_index()
            )
            run.log_figure("seasonal_strength", seasonal_figure)
            run.log_figure("macro_sensitivity_by_sector", sensitivity_figure)

In [ ]:
appendix_c = pd.DataFrame(
    {
        "d_levels": [1, 2, 2, 2, 1, 2],
        "d_log": [0, 2, 1, 1, 2, 2],
        "seasonal_strength": [0.933, 0.908, 0.279, 0.770, 0.961, 0.577],
    },
    index=["AAPL", "AMZN", "GOOGL", "MSFT", "PEP", "PG"],
)
comparison = revenue.loc[
    appendix_c.index, ["d_levels", "d_log", "seasonal_strength"]
].join(appendix_c, rsuffix="_appendix_c")
comparison["seasonal_gap"] = (
    comparison["seasonal_strength"] - comparison["seasonal_strength_appendix_c"]
).abs()
assert comparison["seasonal_gap"].le(0.05).all()  # A3
comparison.round(3)

### A3 — agreement with master plan Appendix C

Seasonal strength matches Appendix C to three decimals for all six companies, and `d_levels` matches for all six. `d_log` matches for five.

**AAPL `d_log`: 1 here, 0 in Appendix C.** On AAPL log revenue at d = 0, ADF rejects a unit root (p = 0.034) but KPSS rejects stationarity (p = 0.01, clipped at the table bound). Spec R3 requires both tests to agree, so the rule moves on to d = 1. Appendix C's 0 is what ADF alone gives. The difference is the decision rule, not the data; it is recorded here, not tuned away (spec A3).

TSLA is left out on purpose: Appendix C's n = 75 came from dropping NaN across gaps, which R2 forbids.

## 3. Feature engineering: lags, seasonal terms, macro joins

All eight groups (L, R, M, X, XD, C, S, F). First prove no feature reads a quarter after its origin (A4), then that every target arm inverts back to the actual level (A5).
For a group ablation, take a service with its own spec: `container.feature_service(builder__spec=FeatureSpec(groups=...))`.

In [ ]:
for horizon in (1, 2, 3, 4):
    for panel in panels.values():
        feature_service.check_lookahead(
            panel, horizon=horizon, origins=[8, 20, 40, 60], regimes=regimes
        )  # A4

In [ ]:
apple_revenue = panels["AAPL"]["revenue_usd_m"]
for arm in TargetArm:
    for horizon in (1, 2, 3, 4):
        transformer = TargetTransformer(arm=arm, horizon=horizon)
        rebuilt = transformer.reconstruct(
            apple_revenue, transformer.make(apple_revenue)
        )
        actual = apple_revenue.shift(-horizon)
        known = rebuilt.notna() & actual.notna()
        np.testing.assert_allclose(rebuilt[known], actual[known], rtol=1e-9)  # A5

## 4. Export features

One parquet per horizon, target arm `seasnaive_residual`, every origin row kept: `y` is NaN for each company's last `h` quarters (spec R8).

In [ ]:
ARM = TargetArm.SEASNAIVE_RESIDUAL
NON_FEATURE_COLUMNS = (
    "ticker",
    "origin_date",
    "target_date",
    "target_level",
    "y",
)
sample_ticker = next(iter(panels))
n_features = feature_service.assemble(
    {sample_ticker: panels[sample_ticker]}, horizon=1, arm=ARM, regimes=regimes
).shape[1] - len(NON_FEATURE_COLUMNS)

config = RunConfig(
    panel_size=len(panels),
    n_rows=n_rows,
    target_variable="revenue_usd_m",
    target_transform=ARM.value,
    feature_groups=tuple(sorted(group.value for group in FeatureGroup)),
    n_features=n_features,
)
with tracker.start_run(
    run_name(notebook="nb02", model="features", variable="revenue_usd_m"),
    config,
    job_type="features",
) as run:
    for horizon in (1, 2, 3, 4):
        dataset = feature_service.assemble(
            panels, horizon=horizon, arm=ARM, regimes=regimes
        )
        assert len(dataset) == n_rows  # A6
        assert dataset["y"].notna().sum() <= n_rows - len(panels) * horizon  # A6
        run.log_dataset(f"features_h{horizon}", feature_store.write(dataset, horizon))